# Day 6: Project Bridge

Transition from refresher exercises to the gaze detection project.

Exercises:
1. Data loading — PyTorch Dataset + DataLoader for face images
2. Model 2 — Frozen MobileNetV2 + regression head
3. Training — MSE loss on yaw/pitch
4. Profiling — torchinfo for parameter counts and FLOPs
5. Export — ONNX sanity check

In [37]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
import torchvision.io as tv_io

---
## Exercise 1: Data Loading

Write a PyTorch Dataset for face images with yaw/pitch labels.

In [4]:
import os                                                                                                                                                      
os.environ["KAGGLEHUB_CACHE"] = "/Volumes/T7/kaggle_datasets"
                                                                                                                                                             
import kagglehub                                                                                                                                               
path = kagglehub.dataset_download("maulidio16/300w-lp")                                                                                                        
print("Path to dataset files:", path)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2.64G/2.64G [01:29<00:00, 31.5MB/s]

Extracting files...


Path to dataset files: /Volumes/T7/kaggle_datasets/datasets/maulidio16/300w-lp/versions/1


In [7]:
# this is a subset of the data of about ~5k images
DATASET_PATH = "/Volumes/T7/kaggle_datasets/datasets/maulidio16/300w-lp/versions/1/300W_LP/AFW"


# experiment with a single file
import scipy.io as sio                                                                                                                                       
mat = sio.loadmat(DATASET_PATH + '/AFW_134212_1_0.mat')                                                                                                                          
pose = mat['Pose_Para'][0]                                                                                                                                   
pitch, yaw, roll = pose[0], pose[1], pose[2]
# Convert to degrees: pitch * 180 / np.pi   
print(f"Pitch: {pitch}")
print(f"Yaw: {yaw}")
print(f"Roll: {roll}")

Pitch: -0.3176567256450653
Yaw: 0.4689221680164337
Roll: -0.15036292374134064


In [50]:
# Create a custom Dataset class:
#   - __init__: load image paths and yaw/pitch labels
#   - __len__: return number of samples
#   - __getitem__: load image, apply transforms, return (image_tensor, label_tensor)
#
# The label should be a tensor of shape (2,): [yaw, pitch]
#
# If you don't have 300W-LP downloaded yet, create a dummy dataset:
#   - 500 random 112x112x3 images
#   - Random yaw/pitch labels in [-90, 90] degrees
#   This lets you build the pipeline without waiting for data
#
# Wrap in a DataLoader with batch_size=32, shuffle=True
# Verify: grab one batch and print shapes

# Your code here:

class MyDataset(Dataset):

    def __init__(self, data_path, transform=None):
        # init the dataset attributes here
        self.data_path = data_path
        self.transform = transform
        self.samples = []

        path = Path(data_path)
        for mat_file in sorted(path.glob("*.mat")):
            img_file = mat_file.with_suffix(".jpg")
            if not img_file.exists():
                continue
            
            mat = sio.loadmat(mat_file)                                                                                                                          
            pose = mat['Pose_Para'][0]                                                                                                                                   
            pitch, yaw = pose[0], pose[1]
            pitch = pitch * 180 / np.pi
            yaw = yaw * 180 / np.pi

            # create a tensor for this lable
            label = torch.tensor([yaw, pitch], dtype=torch.float32)

            self.samples.append((img_file, label))
        
                
        

    def __len__(self):
        # len overload here
        return len(self.samples)

    def __getitem__(self, idx):
        # getitem overload here
        img_path, label = self.samples[idx]
        # this should return a tuple (<image_tensor>, <label_tensor>)
        image_tensor = tv_io.read_image(img_path)
        if self.transform:
            image_tensor = self.transform(image_tensor)
        return (image_tensor, label)


dataset = MyDataset(DATASET_PATH)
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

data_iter = iter(data_loader)
images_batch1, labels_batch1 = next(data_iter)

print(f"images_batch1.shape: {images_batch1.shape:}")
print(f"labels_batch1.shape: {labels_batch1.shape:}")

print(f"dataset[5]: {dataset[5]}")


images_batch1.shape: torch.Size([32, 3, 450, 450])
labels_batch1.shape: torch.Size([32, 2])
dataset[5]: (tensor([[[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 5, 1, 0],
         [0, 0, 0,  ..., 1, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 2, 0, 0],
         [0, 0, 0,  ..., 2, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 4, 0, 0],
         [0, 0, 0,  ..., 3, 2, 0]]], dtype=torch.uint8), tensor([65.2395, -6.7469]))


## Exercise 2: Model 2 — Frozen MobileNetV2 + Regression Head



This is the first real model from the project README.

In [51]:
# 1. Load pretrained MobileNetV2
# 2. Freeze all backbone parameters
# 3. Replace the classifier with a regression head:
#    nn.Sequential(
#        nn.Dropout(0.2),
#        nn.Linear(1280, 2)  # 2 outputs: yaw, pitch
#    )
# 4. Verify only the head parameters have requires_grad=True
# 5. Print total params vs trainable params
#
# Note: MobileNetV2 expects 224x224 input. For 112x112 input,
# either resize to 224 in transforms or the adaptive pool will handle it.

# Your code here:
transformations = transforms.Compose([                                                                                                                     
    transforms.Resize(224),                                                                                                                                  
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])                                                                               
])                  


dataset = MyDataset(DATASET_PATH, transform=transformations)

train_subset, val_subset = random_split(dataset, [0.8, 0.2])
                                                
train_loader = DataLoader(train_subset, batch_size=16, shuffle=True)
test_loader = DataLoader(val_subset, batch_size=16)

model = models.mobilenet_v2(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False


# Replace the classifier with a regression head
model.classifier = nn.Sequential(
    nn.Dropout(.2),
    nn.Linear(1280, 2)
)

# Verify only the head parameters have requires_grad=True
total_params = 0
trainable_params = 0
for param in model.parameters():
    if param.requires_grad:
        trainable_params += 1
    total_params += 1

print(f"Total Params: {total_params}")
print(f"Trainable Param: {trainable_params}")


Total Params: 158
Trainable Param: 2


## Exercise 3: Training with MSE Loss

Train the regression head on face data.

In [52]:
# Training loop:
#   loss_fn = nn.MSELoss()
#   optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
#   (only optimize the head since backbone is frozen)
#
# Train for 10 epochs
# Each epoch: print average MSE loss and MAE (mean absolute error) in degrees
#   MAE = mean(|y_pred - y_true|) — this is the project's primary metric
#
# If using dummy data, the MAE won't be meaningful but the pipeline should work

# Your code here:

# Model setup
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS device found!")
else:
    device = torch.device("cpu")
    print("MPS not available. Using CPU.")

model.to(device)

model.train()

# Training setup
EPOCHS = 10
loss_fnc = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(EPOCHS):
    total_mse = 0.0
    total_mae = 0.0
    num_samples = 0
    for i, (images, labels) in enumerate(train_loader):
        # move the tensors to the gpu (mps)
        images = images.to(device)
        labels = labels.to(device)
        
        # perform inference
        y_hat = model(images)
        
        # compute loss
        loss = loss_fnc(y_hat, labels)
        total_mse += loss.item() * labels.size(0)
        total_mae += torch.abs(y_hat - labels).detach().sum().item()
        num_samples += 1        
        
        if i % 100 == 0:
              print(f"Epoch {epoch}, batch {i}/{len(train_loader)}, loss: {loss.item():.4f}")
        
        # back propagation
        optimizer.zero_grad()  # zeroes out the gradients -> pytorch accumulates the gradients so we have to zero them
        loss.backward()        # perform back propagation -> calculates the gradients
        optimizer.step()       # updates the parameters
    print(f"Epoch {epoch} — MSE: {total_mse / num_samples:.4f}, MAE: {total_mae / (num_samples * 2):.4f}°")


MPS device found!
Epoch 0, batch 0/261, loss: 1687.6671
Epoch 0, batch 100/261, loss: 1816.5413
Epoch 0, batch 200/261, loss: 1490.0284
Epoch 0 — MSE: 25735.4141, MAE: 472.7084°
Epoch 1, batch 0/261, loss: 1814.1783
Epoch 1, batch 100/261, loss: 1584.5295
Epoch 1, batch 200/261, loss: 1936.1943
Epoch 1 — MSE: 24990.3250, MAE: 461.0920°
Epoch 2, batch 0/261, loss: 1426.4202
Epoch 2, batch 100/261, loss: 1251.3308
Epoch 2, batch 200/261, loss: 1826.6693
Epoch 2 — MSE: 24325.5011, MAE: 452.5180°
Epoch 3, batch 0/261, loss: 1461.2266
Epoch 3, batch 100/261, loss: 1387.0212
Epoch 3, batch 200/261, loss: 1718.8673
Epoch 3 — MSE: 23866.1095, MAE: 446.5584°
Epoch 4, batch 0/261, loss: 1202.5715
Epoch 4, batch 100/261, loss: 1662.4871
Epoch 4, batch 200/261, loss: 1616.5789
Epoch 4 — MSE: 23385.7143, MAE: 440.6453°
Epoch 5, batch 0/261, loss: 1143.4139
Epoch 5, batch 100/261, loss: 1646.1163
Epoch 5, batch 200/261, loss: 1110.5675
Epoch 5 — MSE: 22925.2067, MAE: 434.1078°
Epoch 6, batch 0/261, 

## Exercise 4: Model Profiling

Use torchinfo to verify parameter counts and estimate FLOPs.

In [53]:
# pip install torchinfo (if not installed)
from torchinfo import summary

# Run summary on your model with input_size=(1, 3, 112, 112)
# Check:
#   - Total parameters vs trainable parameters (should match exercise 2)
#   - Total mult-adds (FLOPs) — compare to your Day 3 manual calculations
#   - Model size in MB

# Your code here:
summary(model, input_size=(1, 3, 112, 112))


Layer (type:depth-idx)                             Output Shape              Param #
MobileNetV2                                        [1, 2]                    --
├─Sequential: 1-1                                  [1, 1280, 4, 4]           --
│    └─Conv2dNormActivation: 2-1                   [1, 32, 56, 56]           --
│    │    └─Conv2d: 3-1                            [1, 32, 56, 56]           (864)
│    │    └─BatchNorm2d: 3-2                       [1, 32, 56, 56]           (64)
│    │    └─ReLU6: 3-3                             [1, 32, 56, 56]           --
│    └─InvertedResidual: 2-2                       [1, 16, 56, 56]           --
│    │    └─Sequential: 3-4                        [1, 16, 56, 56]           (896)
│    └─InvertedResidual: 2-3                       [1, 24, 28, 28]           --
│    │    └─Sequential: 3-5                        [1, 24, 28, 28]           (5,136)
│    └─InvertedResidual: 2-4                       [1, 24, 28, 28]           --
│    │    └─Sequential

## Exercise 5: ONNX Export

Export the trained model to ONNX and verify it works.

In [59]:
# 1. Export to ONNX:
#    dummy_input = torch.randn(1, 3, 112, 112)
#    torch.onnx.export(model, dummy_input, "model2.onnx")
#
# 2. Reload with ONNX Runtime and verify predictions match:
#    import onnxruntime as ort
#    session = ort.InferenceSession("model2.onnx")
#    ort_input = {session.get_inputs()[0].name: dummy_input.numpy()}
#    ort_output = session.run(None, ort_input)
#    Compare ort_output[0] with model(dummy_input).detach().numpy()
#
# This is the first step of the export pipeline:
# PyTorch → ONNX → (later) TFLite for Raspberry Pi

# Your code here:

# 1
model.eval()
dummy_input = torch.randn(1, 3, 112, 112)
torch.onnx.export(model, dummy_input, "model2.onnx")


W0801 15:27:30.481000 20302 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0801 15:27:30.482000 20302 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0801 15:27:30.482000 20302 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0801 15:27:30.482000 20302 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spat

[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.9.0',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[1,3,112,112]>
            ),
            outputs=(
                %"linear"<FLOAT,[1,2]>
            ),
            initializers=(
                %"features.0.0.weight"<FLOAT,[32,3,3,3]>{Tensor(...)},
                %"features.1.conv.0.0.weight"<FLOAT,[32,1,3,3]>{Tensor(...)},
                %"features.1.conv.1.weight"<FLOAT,[16,32,1,1]>{Tensor(...)},
                %"features.2.conv.0.0.weight"<FLOAT,[96,16,1,1]>{Tensor(...)},
                %"features.2.conv.1.0.weight"<FLOAT,[96,1,3,3]>{Tensor(...)},
                %"features.2.conv.2.weight"<FLOAT,[24,96,1,1]>{Tensor(...)},
                %"features.3.conv.0.0.weight"<FLOAT,[144,24,1,1]>{Tensor(...

In [60]:
import onnxruntime as ort

session = ort.InferenceSession("model2.onnx")
ort_input = {session.get_inputs()[0].name: dummy_input.numpy()}
ort_output = session.run(None, ort_input)


#    Compare ort_output[0] with model(dummy_input).detach().numpy()
print(f"ort_output[0]: {ort_output[0]}")
print(f"model(dummy_input).detach().numpy(): {model(dummy_input).detach().numpy()}")

ort_output[0]: [[-10.270818   -7.1064878]]
model(dummy_input).detach().numpy(): [[-10.270817  -7.106493]]
